# Amazon Reviews Sentiment Training in SageMaker Studio

This notebook performs the complete training process inside the SageMaker Studio notebook kernel. It downloads Glue Gold Parquet data from S3, trains a TF-IDF and Logistic Regression model, evaluates it, and uploads the model artifacts to S3.

## 1. Install the required ML libraries

Run this once when starting a new Studio environment.

In [ ]:
%pip install -q pandas pyarrow scikit-learn joblib

## 2. Import libraries and configure S3

Change the bucket name only if your project uses a different bucket.

In [ ]:
import json
import tarfile
from datetime import datetime, timezone
from pathlib import Path

import boto3
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

bucket = "amazon-food-reviews-ml-model"
gold_prefix = "gold/"
model_prefix = "models/"

boto_session = boto3.Session()
region = boto_session.region_name
s3 = boto_session.client("s3")
run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")

print(f"Region: {region}")
print(f"Gold data: s3://{bucket}/{gold_prefix}")

## 3. Download the Gold Parquet files from S3

In [ ]:
local_gold_dir = Path("data/gold") / run_id
local_gold_dir.mkdir(parents=True, exist_ok=True)

gold_keys = []
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=gold_prefix):
    for item in page.get("Contents", []):
        if item["Key"].endswith(".parquet"):
            gold_keys.append(item["Key"])

if not gold_keys:
    raise FileNotFoundError(f"No Parquet files found at s3://{bucket}/{gold_prefix}")

for key in gold_keys:
    relative_path = Path(key[len(gold_prefix):])
    local_path = local_gold_dir / relative_path
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(bucket, key, str(local_path))

print(f"Downloaded {len(gold_keys)} Parquet file(s) to {local_gold_dir}")

## 4. Load and validate the Gold dataset

In [ ]:
parquet_files = sorted(local_gold_dir.rglob("*.parquet"))
frames = [
    pd.read_parquet(path, columns=["clean_text", "label"])
    for path in parquet_files
]
data = pd.concat(frames, ignore_index=True)

data = data.dropna(subset=["clean_text", "label"]).copy()
data["clean_text"] = data["clean_text"].astype(str).str.strip()
data["label"] = pd.to_numeric(data["label"], errors="coerce")
data = data.dropna(subset=["label"])
data["label"] = data["label"].astype(int)
data = data[
    data["clean_text"].ne("") & data["label"].isin([0, 1])
].reset_index(drop=True)

if data.empty or data["label"].nunique() != 2:
    raise ValueError("Gold data must contain valid clean_text values and both labels 0 and 1")

print(f"Valid records: {len(data):,}")
display(data["label"].value_counts().sort_index().rename("count").to_frame())
display(data.head())

## 5. Split the dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    data["clean_text"],
    data["label"],
    test_size=0.2,
    random_state=42,
    stratify=data["label"],
)

print(f"Training records: {len(X_train):,}")
print(f"Testing records: {len(X_test):,}")

## 6. Create and train the model

This cell performs the training directly on the SageMaker Studio kernel.

In [ ]:
model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                max_features=50000,
                ngram_range=(1, 2),
                min_df=2,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                C=1.0,
                class_weight="balanced",
                max_iter=1000,
                random_state=42,
            ),
        ),
    ]
)

model.fit(X_train, y_train)
print("Training completed")

## 7. Evaluate the model

In [ ]:
predictions = model.predict(X_test)

metrics = {
    "accuracy": float(accuracy_score(y_test, predictions)),
    "precision": float(precision_score(y_test, predictions, zero_division=0)),
    "recall": float(recall_score(y_test, predictions, zero_division=0)),
    "f1": float(f1_score(y_test, predictions, zero_division=0)),
    "confusion_matrix": confusion_matrix(y_test, predictions).tolist(),
    "training_rows": len(X_train),
    "testing_rows": len(X_test),
}

display(pd.DataFrame([{key: value for key, value in metrics.items() if isinstance(value, float)}]))
print("Confusion matrix:")
display(pd.DataFrame(metrics["confusion_matrix"], index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"]))
print(classification_report(y_test, predictions, zero_division=0))

## 8. Save locally and upload the model to S3

In [ ]:
artifact_dir = Path("artifacts/model")
artifact_dir.mkdir(parents=True, exist_ok=True)
model_file = artifact_dir / "model.joblib"
metrics_file = artifact_dir / "metrics.json"
archive_file = artifact_dir / "model.tar.gz"

joblib.dump(model, model_file)
with metrics_file.open("w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

with tarfile.open(archive_file, "w:gz") as archive:
    archive.add(model_file, arcname="model.joblib")
    archive.add(metrics_file, arcname="metrics.json")

s3_artifact_key = f"{model_prefix}studio-{run_id}/model.tar.gz"
s3.upload_file(str(archive_file), bucket, s3_artifact_key)

model_s3_uri = f"s3://{bucket}/{s3_artifact_key}"
print(f"Local model: {model_file}")
print(f"S3 model artifact: {model_s3_uri}")

## 9. Test a prediction

In [ ]:
sample_reviews = [
    "This product is excellent and I really enjoyed using it",
    "This was a terrible purchase and a complete waste of money",
]
sample_predictions = model.predict(sample_reviews)
sample_probabilities = model.predict_proba(sample_reviews)[:, 1]

pd.DataFrame(
    {
        "review": sample_reviews,
        "predicted_label": sample_predictions,
        "positive_probability": sample_probabilities,
    }
)